In [0]:
#Notebook 01: Extract from Source, Write to Staging

from datetime import datetime

#Get parameters from job parameters
catalog_name    = dbutils.widgets.get("catalog_name")
source_catalog  = dbutils.widgets.get("source_catalog")
source_schema   = dbutils.widgets.get("source_schema")
meta_schema     = dbutils.widgets.get("meta_schema")
staging_schema  = dbutils.widgets.get("staging_schema")

#Reading all active tables from metadata
active_tables_df = spark.sql(f"""
    SELECT table_name 
    FROM {catalog_name}.{meta_schema}.parent_metadata
    WHERE active_flag = 'Y'
""")

active_tables = [row["table_name"] for row in active_tables_df.collect()]
print(f"Active tables: {active_tables}")

#Extracting each table and writing to staging
for table_name in active_tables:
    try:
        #Extracting data from from Azure SQL
        source_df = spark.sql(f"""
            SELECT * FROM {source_catalog}.{source_schema}.{table_name}
        """)
        
        source_row_count = source_df.count()

        #Writing to staging table (overwrites in each run)
        source_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(f"{catalog_name}.{staging_schema}.{table_name.lower()}")

        print(f"{table_name}: {source_row_count} rows written to staging")

    except Exception as e:
        print(f"{table_name} failed: {str(e)}")

Active tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Album: 347 rows written to staging
Artist: 275 rows written to staging
Customer: 59 rows written to staging
Employee: 8 rows written to staging
Genre: 25 rows written to staging
Invoice: 412 rows written to staging
InvoiceLine: 2240 rows written to staging
MediaType: 5 rows written to staging
Playlist: 18 rows written to staging
PlaylistTrack: 8715 rows written to staging
Track: 3503 rows written to staging
